In [ ]:
from myutils import *
from src import *

In [ ]:
class DI(_Share):
    SIGMA = 103 #um

    X_AXIS = 82
    CENTER = 109
    
    ROI = {'X0': 1440, 'Y0': 880, 'W': 160, 'H': 220}

    def __init__(self, raw, amplitude):
        super().__init__(raw)
        self.amplitude = amplitude
        self.upper_bound = int(np.ceil(self.CENTER - (2*amplitude + 3*self.SIGMA) / qCMOS.PIXEL_SIZE))
        self.lower_bound = int(np.ceil(self.CENTER + 3*self.SIGMA / qCMOS.PIXEL_SIZE))
        self.detectors = self.lower_bound - self.upper_bound

    def crop(self):
        self.cropped = self.raw[..., self.upper_bound:self.lower_bound, self.X_AXIS]

In [ ]:

def FrequencyEstmation(raw: np.ndarray, metadata: MetaData):
    if metadata.measurement.upper() == 'SPADE':
        expt = SPADE(raw)
    elif metadata.measurement.upper() == 'DI':
        expt = DI(raw, metadata.amplitude)
    else:
        raise ValueError
    
    expt.est_all()
    return Estmates(frequency_estmates = expt.lse[..., 0], 
                    phase_estmates = expt.lse[..., 1], 

                    cropped_data = expt.cropped, 
                    time_domain = expt.td, 
                    photons = expt.pn, 

                    noise = expt.noise,
                    noise_weight = expt.w,
                
                    metadata = metadata)
